# RQ2 deep learning on Google Colab

RQ2 trains LSTM, GRU, and CNN models and needs an NVIDIA GPU. The reported results were run on Colab with PyTorch 2.11.0.

**1. GPU:** `Runtime → Change runtime type → GPU`.

**2. Upload these 6 files** from the package to `MyDrive/DSRT930_RQ2/` in Google Drive:
```
data/final/Final_Paired_Email_Dataset.csv
data/final/With_Bulk_Emails_Dataset.csv
data/final/Masked_Identifier_Terms.csv
code/rq_scripts/splits.py
code/rq_scripts/sensitivity.py
code/rq_scripts/RQ2_deep_learning/scripts/run_rq2_deep_learning.py
```

**3. Run:** set `RUN` in the second code cell, then `Runtime → Run all`. Progress is saved to Drive after every iteration; if Colab disconnects, run all again with the same `RUN` and it continues.

| RUN | Analysis (Table 2) | Iterations |
|---|---|---|
| `primary_grouped_split` | Primary (grouped split) | 500 |
| `random_split` | Random split | 100 |
| `masked_identifiers` | Masked identifiers | 100 |
| `with_bulk_emails` | With bulk emails | 100 |
| `without_repeated_passage_pairs` | Without repeated-passage pairs | 100 |

**4. When a run finishes**, download its output folder (path printed by the last cell) into
`<rerun workspace>/rq_scripts/RQ2_deep_learning/output/` on your computer. After all five, run
`python run_all.py --stage rq3` and `python run_all.py --stage outlier`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, shutil

FOLDER = Path('/content/drive/MyDrive/DSRT930_RQ2')
NEEDED = ['Final_Paired_Email_Dataset.csv', 'With_Bulk_Emails_Dataset.csv', 'Masked_Identifier_Terms.csv',
          'splits.py', 'sensitivity.py', 'run_rq2_deep_learning.py']
EXPECTED_SHA = {
    'Final_Paired_Email_Dataset.csv': '25dcf2247880d249e5bd188c5c7624b843764838dc39c476543ad5ada86ac47d',
    'With_Bulk_Emails_Dataset.csv': '2fdc429d04576f2d41fa55a71f8049a6f92dd67fa9e9ad68b139bb7a13095b55',
    'Masked_Identifier_Terms.csv': 'bf30e114399f013ea91256756556a7804185006d5de5c82b0bb2019b6af87e68',
}
found = {}
for name in NEEDED:
    hits = [h for h in sorted(FOLDER.rglob(name)) if 'work' not in h.parts]
    ok = len(hits) == 1
    detail = '' if ok else ('  -> not found' if not hits else '  -> several copies: ' + ', '.join(map(str, hits)))
    if ok and name in EXPECTED_SHA and hashlib.sha256(hits[0].read_bytes()).hexdigest() != EXPECTED_SHA[name]:
        ok, detail = False, '  -> not the frozen version; upload it again'
    print(('OK       ' if ok else 'PROBLEM  ') + name + detail)
    if ok:
        found[name] = hits[0]
assert len(found) == len(NEEDED), 'Fix the files marked PROBLEM in Google Drive, then run again.'

# Same layout as the package, on Drive, so outputs and checkpoints survive a disconnect.
WORK = FOLDER / 'work'
SCRIPTS = WORK / 'rq_scripts' / 'RQ2_deep_learning' / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)
for name in ['splits.py', 'sensitivity.py']:
    shutil.copy(found[name], WORK / 'rq_scripts' / name)
shutil.copy(found['run_rq2_deep_learning.py'], SCRIPTS)
print('Ready.')

In [ ]:
RUN = 'primary_grouped_split'   # primary_grouped_split | random_split | masked_identifiers | with_bulk_emails | without_repeated_passage_pairs

FINAL = str(found['Final_Paired_Email_Dataset.csv'])
COMMON = ['--device', 'cuda', '--resume', '--checkpoint-every', '1', '--run-label', RUN]
RUNS = {
    'primary_grouped_split':          ['--split', 'grouped', '--iterations', '500', '--master-path', FINAL],
    'random_split':                   ['--split', 'random', '--iterations', '100', '--master-path', FINAL],
    'masked_identifiers':             ['--split', 'grouped', '--iterations', '100', '--master-path', FINAL,
                                       '--mask-terms', str(found['Masked_Identifier_Terms.csv'])],
    'with_bulk_emails':               ['--split', 'grouped', '--iterations', '100',
                                       '--master-path', str(found['With_Bulk_Emails_Dataset.csv'])],
    'without_repeated_passage_pairs': ['--split', 'grouped', '--iterations', '100', '--master-path', FINAL,
                                       '--exclude-pair-ids', 'nazario_001874,nazario_000153'],
}
ARGS = RUNS[RUN] + COMMON
print('Run:', RUN)
print(' '.join(ARGS))

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to GPU first (Runtime -> Change runtime type).'

In [ ]:
import subprocess, sys
cmd = [sys.executable, str(SCRIPTS / 'run_rq2_deep_learning.py')] + ARGS
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith(('Completed iteration', 'Resuming', 'Completed iterations found', 'Traceback')) or 'Error' in line:
        print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)

In [ ]:
import pandas as pd
OUT = WORK / 'rq_scripts' / 'RQ2_deep_learning' / 'output' / RUN
audit = pd.read_csv(OUT / 'RQ2_analysis_audit.csv')
done = int(audit.loc[audit['item'].eq('completed_iterations'), 'value'].iloc[0])
want = int(audit.loc[audit['item'].eq('requested_iterations'), 'value'].iloc[0])
print(f'{RUN}: {done} / {want} iterations complete. Output folder in Google Drive: {OUT}')
print('Finished - download this folder (step 4 above).' if done == want else 'Not finished - run all again to continue.')